<a href="https://colab.research.google.com/github/serkansentilay/dogaldilisleme/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers torch pandas scikit-learn matplotlib seaborn -q

In [2]:
import pandas as pd

df = pd.read_csv('/content/909090.csv',
                 sep=';',
                 encoding='utf-8',
                 on_bad_lines='skip',
                 skiprows=1)

df.columns = ['singer', 'lyrics']

print("Satır sayısı:", len(df))
df.head()

Satır sayısı: 5776


,singer,lyrics
0,Ahmet Kaya,Bir mavi otobüs gelirdi\nSeni alır giderdi\nO ...
1,Ahmet Kaya,"Uyusun ha iyi büyüsün,\nCamlar buğulanmasın\nS..."
2,Ahmet Kaya,Kavuşmak özgürlükse özgürdük ikimiz de\nElleri...
3,Ahmet Kaya,Ada sahillerinde bekliyorum\nHer zaman yolları...
4,Ahmet Kaya,"Geçiyor önümden, sirenler içinde\nAh eller üst..."


In [3]:
# Temizleme
df = df.dropna(subset=['lyrics'])
df = df[df['lyrics'].str.strip() != '']

# \n karakterlerini boşluğa çevir
df['lyrics'] = df['lyrics'].str.replace('\\n', ' ', regex=False)
df['lyrics'] = df['lyrics'].str.replace('\n', ' ', regex=False)

# Kelime sayısı ekle ve çok kısaları at
df['word_count'] = df['lyrics'].apply(lambda x: len(str(x).split()))
df = df[df['word_count'] >= 10].reset_index(drop=True)

# Model için kısaltılmış versiyon (max 400 kelime)
df['lyrics_short'] = df['lyrics'].apply(lambda x: ' '.join(str(x).split()[:400]))

print("Temiz satır sayısı:", len(df))
print("\nSanatçılar:")
print(df['singer'].value_counts().head(10))
df.head(3)

Temiz satır sayısı: 5773

Sanatçılar:
singer
Zeki Müren        505
Müslüm Gürses     454
Sezen Aksu        369
Sagopa Kajmer     326
Orhan Gencebay    308
Serdar Ortaç      257
Ahmet Kaya        243
Ceza              222
Sertab Erener     161
Barış Manço       161
Name: count, dtype: int64


,singer,lyrics,word_count,lyrics_short
0,Ahmet Kaya,Bir mavi otobüs gelirdi Seni alır giderdi O ma...,85,Bir mavi otobüs gelirdi Seni alır giderdi O ma...
1,Ahmet Kaya,"Uyusun ha iyi büyüsün, Camlar buğulanmasın Sen...",45,"Uyusun ha iyi büyüsün, Camlar buğulanmasın Sen..."
2,Ahmet Kaya,Kavuşmak özgürlükse özgürdük ikimiz de Elleri ...,132,Kavuşmak özgürlükse özgürdük ikimiz de Elleri ...


In [4]:
from transformers import pipeline
from tqdm import tqdm
tqdm.pandas()

# Türkçe + çok dilli zero-shot model
genre_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)
# Daha açıklayıcı etiketler - model bunları daha iyi anlıyor
GENRES = [
    "arabesk music with sad emotional Turkish lyrics about love and pain",
    "Turkish pop music with modern upbeat rhythm",
    "rock music with electric guitar and energetic lyrics",
    "Turkish rap and hip hop with fast rhyming lyrics",
    "Turkish folk music with traditional cultural themes",
    "Turkish classical art music with Ottoman influence",
    "slow romantic Turkish ballad about longing and heartbreak",
    "electronic dance music with synthesizer beats"
]

# Kısa etiket eşleştirmesi (gösterim için)
GENRE_LABELS = {
    "arabesk music with sad emotional Turkish lyrics about love and pain": "arabesk",
    "Turkish pop music with modern upbeat rhythm": "pop",
    "rock music with electric guitar and energetic lyrics": "rock",
    "Turkish rap and hip hop with fast rhyming lyrics": "rap/hip-hop",
    "Turkish folk music with traditional cultural themes": "halk müziği",
    "Turkish classical art music with Ottoman influence": "sanat müziği",
    "slow romantic Turkish ballad about longing and heartbreak": "slow/romantik",
    "electronic dance music with synthesizer beats": "elektronik"
}

def predict_genre(lyrics):
    try:
        result = genre_classifier(
            lyrics[:512],
            candidate_labels=GENRES,
            multi_label=False
        )
        full_label = result['labels'][0]
        short_label = GENRE_LABELS.get(full_label, full_label)
        return short_label, round(result['scores'][0], 3)
    except:
        return "bilinmiyor", 0.0

# Test
for i in range(5):
    genre, score = predict_genre(df['lyrics_short'].iloc[i])
    print(f"{df['singer'].iloc[i]:20s} → {genre} ({score})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ahmet Kaya           → halk müziği (0.204)
Ahmet Kaya           → halk müziği (0.236)
Ahmet Kaya           → sanat müziği (0.169)
Ahmet Kaya           → sanat müziği (0.215)
Ahmet Kaya           → halk müziği (0.246)


In [5]:
# Sanatçı bazlı tür sözlüğü - veri setindeki sanatçılara göre
SINGER_GENRE_MAP = {
    # Arabesk
    'Zeki Müren': 'sanat müziği',
    'Müslüm Gürses': 'arabesk',
    'Orhan Gencebay': 'arabesk',
    'Ibrahim Tatlises': 'arabesk',
    'Ferdi Tayfur': 'arabesk',

    # Pop
    'Sezen Aksu': 'pop',
    'Serdar Ortaç': 'pop',
    'Sertab Erener': 'pop',
    'Tarkan': 'pop',
    'Hadise': 'pop',

    # Rap/Hip-hop
    'Sagopa Kajmer': 'rap',
    'Ceza': 'rap',

    # Halk/Rock
    'Barış Manço': 'halk rock',
    'Ahmet Kaya': 'halk rock',
}

df['genre'] = df['singer'].map(SINGER_GENRE_MAP).fillna('diğer')

print(df['genre'].value_counts())
print("\nEtiketlenemeyen sanatçılar:")
print(df[df['genre'] == 'diğer']['singer'].unique())

genre
diğer           2608
pop              938
arabesk          770
rap              548
sanat müziği     505
halk rock        404
Name: count, dtype: int64

Etiketlenemeyen sanatçılar:
['Büyük Ev Ablukada' 'Can Bonomo' 'Candan Erçetin' 'Cem Adrian'
 'Cem Karaca' 'Duman' 'Ezhel' 'Feridun Düzağaç' 'Kayahan' 'Mabel Matiz'
 'Mazhar Fuat Özkan (MFÖ)' 'Mor Ve Ötesi' 'Mustafa Sandal' 'Nazan Öncel'
 'Pinhani' 'Teoman' 'Yaşar' 'Yeni Türkü' 'Zülfü Livaneli' 'Şebnem Ferah'
 'Azer Bülbül' 'Yıldız Tilbe' 'Cengiz Özkan' 'Sema' 'İbrahim Tatlıses'
 'Karmate' 'Volkan Konak' 'Mümin Sarıkaya' 'Haluk Levent' 'Edip Akbayram'
 'Murat Göğebakan' 'Murat Kekilli' 'Emel Sayın' 'Neşet Ertaş'
 'Muharrem Aslan' 'Ayla Dikmen' 'Özdemir Erdoğan' 'Bergen'
 'Cengiz Kurtoğlu' 'Kamuran Akkor' 'Erkin Koray' 'Güllü' 'Bülent Ersoy'
 'Esmeray' 'Gülden Karaböcek' 'Sinan Özen' 'Kibariye' 'Tülay Özer'
 'Ümit Besen' 'Bülent Ersoy, Tarkan' 'Adnan Senses' 'Arap Şükrü'
 'Murat Yeter, Ebru Gündeş' 'Muazzez Ersoy' 'Belkıs Özener' '

In [6]:
# Bilinen sanatçıları kategorilere göre grupla
SINGER_GENRE_MAP = {
    # Sanat müziği
    'Zeki Müren': 'sanat müziği', 'Müzeyyen Senar': 'sanat müziği',
    'Emel Sayın': 'sanat müziği', 'Bülent Ersoy': 'sanat müziği',
    'Muazzez Ersoy': 'sanat müziği', 'Muazzez Abacı': 'sanat müziği',

    # Arabesk
    'Müslüm Gürses': 'arabesk', 'Orhan Gencebay': 'arabesk',
    'İbrahim Tatlıses': 'arabesk', 'Ferdi Tayfur': 'arabesk',
    'Azer Bülbül': 'arabesk', 'Bergen': 'arabesk',
    'Murat Göğebakan': 'arabesk', 'Cengiz Kurtoğlu': 'arabesk',
    'Sibel Can': 'arabesk', 'Ebru Gündeş': 'arabesk',
    'Yıldız Tilbe': 'arabesk', 'Hakan Altun': 'arabesk',
    'Sinan Özen': 'arabesk', 'Kibariye': 'arabesk',
    'Güllü': 'arabesk', 'Esmeray': 'arabesk',
    'Özdemir Erdoğan': 'arabesk', 'Kamuran Akkor': 'arabesk',
    'Gülden Karaböcek': 'arabesk', 'Ümit Besen': 'arabesk',
    'Adnan Şenses': 'arabesk', 'İbrahim Erkal': 'arabesk',
    'Kubat': 'arabesk', 'Özgün': 'arabesk',
    'Mustafa Ceceli': 'arabesk', 'Semicenk': 'arabesk',
    'Bilal Sonses': 'arabesk', 'Serkan Kaya': 'arabesk',

    # Halk müziği
    'Neşet Ertaş': 'halk müziği', 'Yeni Türkü': 'halk müziği',
    'Zülfü Livaneli': 'halk müziği', 'Cengiz Özkan': 'halk müziği',
    'Volkan Konak': 'halk müziği', 'Muharrem Aslan': 'halk müziği',
    'Grup Yorum': 'halk müziği', 'Grup Laçin': 'halk müziği',
    'Edip Akbayram': 'halk müziği', 'Ankaralı Namık': 'halk müziği',

    # Halk rock
    'Barış Manço': 'halk rock', 'Ahmet Kaya': 'halk rock',
    'Cem Karaca': 'halk rock', 'MFÖ': 'halk rock',
    'Mazhar Fuat Özkan (MFÖ)': 'halk rock', 'Erkin Koray': 'halk rock',
    'Haluk Levent': 'halk rock',

    # Rock
    'Duman': 'rock', 'Mor Ve Ötesi': 'rock', 'mor ve ötesi': 'rock',
    'Yüksek Sadakat': 'rock', 'Şebnem Ferah': 'rock',
    'Murat Kekilli': 'rock', 'Gripin': 'rock', 'Redd': 'rock',
    'Büyük Ev Ablukada': 'rock', 'Adamlar': 'rock',

    # Pop
    'Sezen Aksu': 'pop', 'Serdar Ortaç': 'pop', 'Sertab Erener': 'pop',
    'Tarkan': 'pop', 'Hadise': 'pop', 'Mustafa Sandal': 'pop',
    'Ajda Pekkan': 'pop', 'Candan Erçetin': 'pop', 'Kayahan': 'pop',
    'Nazan Öncel': 'pop', 'Teoman': 'pop', 'Mabel Matiz': 'pop',
    'Can Bonomo': 'pop', 'Cem Adrian': 'pop', 'Pinhani': 'pop',
    'Feridun Düzağaç': 'pop', 'Yaşar': 'pop', 'Göksel': 'pop',
    'Sıla': 'pop', 'Murat Boz': 'pop', 'Kenan Doğulu': 'pop',
    'Gülşen': 'pop', 'Hande Yener': 'pop', 'Demet Akalın': 'pop',
    'Aleyna Tilki': 'pop', 'Oğuzhan Koç': 'pop', 'Atiye': 'pop',
    'Tan Taşçı': 'pop', 'Halil Sezai': 'pop', 'Funda Arar': 'pop',
    'Bengü': 'pop', 'Gökhan Türkmen': 'pop', 'Yalın': 'pop',
    'Fettah Can': 'pop', 'Burak Kut': 'pop', 'Hakan Peker': 'pop',
    'Ezhel': 'pop', 'Zara': 'pop', 'Simge': 'pop',
    'Zeynep Dizdar': 'pop', 'Yonca Evcimik': 'pop',
    'Nil Karaibrahimgil': 'pop', 'Dolu Kadehi Ters Tut': 'pop',
    'Gökhan Özen': 'pop', 'Murat Dalkılıç': 'pop', 'Buray': 'pop',
    'EDIS': 'pop', 'Ece Seçkin': 'pop', 'Aydilge': 'pop',
    'Ozan Doğulu': 'pop', 'Emre Altuğ': 'pop',

    # Rap / Hip-hop
    'Sagopa Kajmer': 'rap', 'Ceza': 'rap', 'Norm Ender': 'rap',
    'Şanışer': 'rap', 'Patron': 'rap', 'UZI': 'rap',
    'Lvbel C5': 'rap', 'Gazapizm': 'rap', 'Heijan': 'rap',
    'Ben Fero': 'rap', 'Sefo': 'rap', 'Khontkar': 'rap',
    'Şehinşah': 'rap', 'Allame': 'rap', 'Hidra': 'rap',
    'Canbay & Wolker': 'rap', 'Tankurt Manas': 'rap',
    'Sansar Salvo': 'rap', 'Cash Flow': 'rap', 'Rota': 'rap',
    'Joker': 'rap', 'Contra': 'rap', 'Anıl Piyancı': 'rap',
    'Mode XL': 'rap', 'Batuflex': 'rap', 'cakal': 'rap',
    'Revart': 'rap', 'Murda': 'rap', 'Ati242': 'rap', 'Ceg': 'rap',
}

# Uygula
df['genre'] = df['singer'].map(SINGER_GENRE_MAP)

# Hala boş kalanlar için singer adından tahmin et
def guess_genre_from_name(singer):
    singer = str(singer).lower()
    rap_keywords = ['uzi', 'flow', 'maho', 'vio', 'bege', 'xir', 'muti',
                    'reckol', 'aspova', 'zen', 'doğu swag', 'kayra']
    if any(k in singer for k in rap_keywords):
        return 'rap'
    return 'pop'  # belirsizler için varsayılan

df['genre'] = df.apply(
    lambda row: row['genre'] if pd.notna(row['genre'])
    else guess_genre_from_name(row['singer']),
    axis=1
)

print("Tür dağılımı:")
print(df['genre'].value_counts())
print(f"\nEtiketlenmemiş kalan: {df['genre'].isna().sum()}")

Tür dağılımı:
genre
pop             2507
arabesk          861
rap              666
halk rock        648
sanat müziği     514
rock             316
halk müziği      261
Name: count, dtype: int64

Etiketlenmemiş kalan: 0


In [7]:
from transformers import MarianMTModel, MarianTokenizer
from transformers import pipeline
from tqdm import tqdm
tqdm.pandas()

# Çeviri modelini direkt yükle
model_name = "Helsinki-NLP/opus-mt-tr-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
translation_model = MarianMTModel.from_pretrained(model_name)

def translate_tr_to_en(text):
    try:
        inputs = tokenizer(text[:400], return_tensors="pt", truncation=True, max_length=512)
        translated = translation_model.generate(**inputs)
        return tokenizer.decode(translated[0], skip_special_tokens=True)
    except:
        return text  # hata olursa orijinali döndür

# 7 kategorili duygu modeli
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

EMOTION_MAP = {
    "joy":      "neşe 🎉",
    "sadness":  "hüzün 💔",
    "anger":    "öfke 😡",
    "fear":     "korku 😨",
    "surprise": "şaşkınlık 😲",
    "disgust":  "tiksinti 🤢",
    "neutral":  "nötr 😶"
}

def predict_emotion(lyrics):
    try:
        translated = translate_tr_to_en(lyrics)
        result = emotion_classifier(translated[:512], truncation=True)[0]
        label = result['label'].lower()
        return EMOTION_MAP.get(label, label), round(result['score'], 3)
    except:
        return "bilinmiyor", 0.0

# 5 şarkıda test et
for i in range(5):
    emotion, score = predict_emotion(df['lyrics_short'].iloc[i])
    print(f"{df['singer'].iloc[i]:20s} → {emotion} ({score})")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ahmet Kaya           → nötr 😶 (0.782)
Ahmet Kaya           → nötr 😶 (0.493)
Ahmet Kaya           → nötr 😶 (0.526)
Ahmet Kaya           → neşe 🎉 (0.488)
Ahmet Kaya           → hüzün 💔 (0.525)


In [8]:
# Her türden eşit örnek al (toplam ~100 şarkı)
df_sample = df.groupby('genre', group_keys=False).apply(
    lambda x: x.sample(min(15, len(x)), random_state=42)
).reset_index(drop=True)

print(f"Örnek boyutu: {len(df_sample)}")
print(df_sample['genre'].value_counts())

Örnek boyutu: 105
genre
arabesk         15
halk müziği     15
halk rock       15
pop             15
rap             15
rock            15
sanat müziği    15
Name: count, dtype: int64


/tmp/ipykernel_21487/3339793723.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('genre', group_keys=False).apply(


In [9]:
print("Duygu analizi yapılıyor (~100 şarkı)...")
df_sample[['predicted_emotion', 'emotion_confidence']] = df_sample['lyrics_short'].progress_apply(
    lambda x: pd.Series(predict_emotion(x))
)

print("\nDuygu dağılımı:")
print(df_sample['predicted_emotion'].value_counts())

print("\nTüre göre duygu:")
print(pd.crosstab(df_sample['genre'], df_sample['predicted_emotion']))

Duygu analizi yapılıyor (~100 şarkı)...


100%|██████████| 105/105 [15:05<00:00,  8.62s/it]


Duygu dağılımı:
predicted_emotion
nötr 😶         31
öfke 😡         27
hüzün 💔        19
korku 😨        10
tiksinti 🤢      8
şaşkınlık 😲     6
neşe 🎉          4
Name: count, dtype: int64

Türe göre duygu:
predicted_emotion  hüzün 💔  korku 😨  neşe 🎉  nötr 😶  tiksinti 🤢  öfke 😡  \
genre                                                                     
arabesk                  6        1       2       2           2       2   
halk müziği              2        1       1       8           2       0   
halk rock                3        2       0       2           0       7   
pop                      1        0       1       8           0       4   
rap                      1        0       0       6           3       4   
rock                     3        4       0       0           1       6   
sanat müziği             3        2       0       5           0       4   

predicted_emotion  şaşkınlık 😲  
genre                           
arabesk                      0  
halk müziği         

In [12]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Modeli direkt yükle
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

def summarize_lyrics(lyrics):
    try:
        # Önce İngilizce'ye çevir
        translated = translate_tr_to_en(lyrics[:400])
        if len(translated.split()) < 20:
            return "Şarkı sözü özet için çok kısa."

        inputs = bart_tokenizer(
            translated[:600],
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        summary_ids = bart_model.generate(
            **inputs,
            max_length=60,
            min_length=15,
            num_beams=4
        )
        return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    except Exception as e:
        return f"Hata: {str(e)[:50]}"

# 5 şarkıda test et
for i in range(5):
    summary = summarize_lyrics(df_sample['lyrics_short'].iloc[i])
    print(f"\n🎵 {df_sample['singer'].iloc[i]} [{df_sample['genre'].iloc[i]}]")
    print(f"😔 {df_sample['predicted_emotion'].iloc[i]}")
    print(f"📝 {summary}")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


🎵 Orhan Gencebay [arabesk]
😔 hüzün 💔
📝 I want a love, a love that has never happened, a promise that never been said, that I may suffer a life that is never said. How all the emotions that have been left without you have gone through my heart for many years only to be satisfied with you. Half the

🎵 Orhan Gencebay [arabesk]
😔 neşe 🎉
📝 I've always been asking my fortune, when I'll love, I'll give you love. My heart is spreading wings, my heart is flying as if I'm happy, my fate is flying. You've given me a light, you've come to my changing world.

🎵 Müslüm Gürses [arabesk]
😔 nötr 😶
📝 He's got a lot on his plate. He'sGot a death wish.

🎵 Müslüm Gürses [arabesk]
😔 öfke 😡
📝 "Why are your silent eyes staring at me like you're hiding something from my eyes?" she asks. "Why are you in love with me?"

🎵 Müslüm Gürses [arabesk]
😔 hüzün 💔
📝 Would you believe me if I told you how much I'd suffered when I left you? I'd be devastated. If I said I loved you again, would you let me burn the flame? I

In [14]:
tr_model_name = "Helsinki-NLP/opus-mt-tc-big-en-tr"
tr_tokenizer = MarianTokenizer.from_pretrained(tr_model_name)
tr_model = MarianMTModel.from_pretrained(tr_model_name)

def translate_en_to_tr(text):
    try:
        inputs = tr_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        translated = tr_model.generate(**inputs)
        return tr_tokenizer.decode(translated[0], skip_special_tokens=True)
    except:
        return text

# Test
print(translate_en_to_tr("I want a love that has never happened before."))

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/833k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Daha önce hiç yaşanmamış bir aşk istiyorum.


In [15]:
def summarize_lyrics(lyrics):
    try:
        translated_en = translate_tr_to_en(lyrics[:400])
        if len(translated_en.split()) < 20:
            return "Şarkı sözü özet için çok kısa."
        inputs = bart_tokenizer(translated_en[:600], return_tensors="pt", truncation=True, max_length=512)
        summary_ids = bart_model.generate(**inputs, max_length=60, min_length=15, num_beams=4)
        summary_en = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        # İngilizce özeti Türkçe'ye çevir
        summary_tr = translate_en_to_tr(summary_en)
        return summary_tr
    except Exception as e:
        return f"Hata: {str(e)[:50]}"

# 5 şarkıda test et
for i in range(5):
    summary = summarize_lyrics(df_sample['lyrics_short'].iloc[i])
    print(f"\n🎵 {df_sample['singer'].iloc[i]} [{df_sample['genre'].iloc[i]}]")
    print(f"😔 {df_sample['predicted_emotion'].iloc[i]}")
    print(f"📝 {summary}")


🎵 Orhan Gencebay [arabesk]
😔 hüzün 💔
📝 Ben bir aşk istiyorum, hiç yaşanmamış bir aşk, hiç söylenmemiş bir söz, asla söylenmemiş bir hayat yaşayabilirim. Sensiz kalan tüm duygular nasıl da yıllarca kalbimden geçip seninle tatmin olmak için gitti. Yarısı

🎵 Orhan Gencebay [arabesk]
😔 neşe 🎉
📝 Her zaman kaderimi sorardım, ne zaman seveceğim, sana sevgi vereceğim. Kalbim kanatlarını açıyor, kalbim mutluymuş gibi uçuyor, kaderim uçuyor. Bana bir ışık verdin, değişen dünyama geldin.

🎵 Müslüm Gürses [arabesk]
😔 nötr 😶
📝 Tabağında çok fazla ölüm arzusu var.

🎵 Müslüm Gürses [arabesk]
😔 öfke 😡
📝 "Neden sessiz gözlerin gözlerimden bir şey saklıyormuş gibi bana bakıyor?" diye soruyor. "Neden bana aşıksın?"

🎵 Müslüm Gürses [arabesk]
😔 hüzün 💔
📝 Seni terk ettiğimde ne kadar acı çektiğimi söylesem inanır mısın? Seni tekrar sevdiğimi söylesem, alevi yakmama izin verir misin? Sevginle, aşkımla yaşadım. Kalbimdeki yeri hazırlar mısın?


In [17]:
import os

# Repo klasöründeki dosyaları listele
for f in os.listdir('/content/dogaldilisleme'):
    print(f)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dogaldilisleme'